<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-geollms/blob/main/phase1_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NDVI dataset preprocessing

In [ ]:
import os
import pandas as pd
import requests
from io import StringIO
import networkx as nx
from scipy.spatial import cKDTree
import folium
from math import sqrt
from geopy.geocoders import Nominatim

# Step 1: Clone the GitHub repository if not already present
repo_url = "https://github.com/Dr-Isam-ALJAWARNEH/fds-project-geollms.git"
clone_dir = "fds-project-geollms"
if not os.path.exists(clone_dir):
    os.system(f"git clone {repo_url}")

# Step 2: Path to NDVI folder
ndvi_folder = os.path.join(clone_dir, "Datasets", "NDVI")

# Step 3: Read and analyze each CSV
all_dfs = []
print("Reading files from NDVI folder...\n")

for file in os.listdir(ndvi_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(ndvi_folder, file)
        df = pd.read_csv(file_path)

        # Check and report missing values in each file
        missing = df.isnull().sum()
        print(f" File: {file}")
        print(f" → Rows: {len(df)}, Columns: {len(df.columns)}")
        print(f" → Missing values:\n{missing}\n")

        all_dfs.append(df)

# Step 4: Combine all into one DataFrame
combined_ndvi_df = pd.concat(all_dfs, ignore_index=True)

# Step 5: Final summary
print(" Combined NDVI Dataset Info:")
print(combined_ndvi_df.info())


AQ Dataset Preprocessing

In [ ]:

# URL for GitHub files
base_url = "https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/AQ_data/"

# List of files to read
filenames = [f"chicago_eclipse_data_part_{i}.csv" for i in range(1, 20)]

# List to store DataFrames
dfs = []

# Download and read each CSV into a DataFrame
for filename in filenames:
    url = base_url + filename
    response = requests.get(url)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        dfs.append(df)
    else:
        print(f"Failed to load: {filename}")

# Combine all CSV files into one DataFrame
aq_df = pd.concat(dfs, ignore_index=True)

# Check for missing values
missing_pm25 = aq_df['PM25'].isnull().sum()
missing_datetime = aq_df['ReadingDateTimeUTC'].isnull().sum()
missing_lat = aq_df['Latitude'].isnull().sum()
missing_lon = aq_df['Longitude'].isnull().sum()

print(f"Missing values in 'PM25': {missing_pm25}")
print(f"Missing values in 'ReadingDateTimeUTC': {missing_datetime}")
print(f"Missing values in 'Latitude': {missing_lat}")
print(f"Missing values in 'Longitude': {missing_lon}")


Main Code

In [ ]:
# Initialize geocoder
geolocator = Nominatim(user_agent="geo_path_finder")

# Function to geocode a place (convert location name to coordinates)
def geocode_place(place_name):
    location = geolocator.geocode(place_name + ", Chicago, IL")  # Restrict search to Chicago
    if location:
        return (location.longitude, location.latitude)
    else:
        raise ValueError(f"Location '{place_name}' could not be found.")

# Step 1: Load datasets from GitHub
bike_url = 'https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/OSM%20datasets/chicago_bike_edges_useful_2.csv'
bike_df = pd.read_csv(bike_url, skipinitialspace=True)

# Step 2: Build the bike network graph
G = nx.Graph()
for _, row in bike_df.iterrows():
    u = (row['start_lon'], row['start_lat'])
    v = (row['end_lon'], row['end_lat'])
    weight = row['length']
    G.add_edge(u, v, length=weight)

# Step 3: Filter AQ data to use only the latest reading per sensor location
aq_df['ReadingDateTimeUTC'] = pd.to_datetime(aq_df['ReadingDateTimeUTC'])
latest_aq_df = aq_df.sort_values('ReadingDateTimeUTC').groupby(['Latitude', 'Longitude']).tail(1)
latest_aq_coords = latest_aq_df[['Latitude', 'Longitude']].values
aq_tree = cKDTree(latest_aq_coords)

 

# Prepare latest NDVI points
# If your NDVI data has a time column like 'Date', uncomment next line
# combined_ndvi_df['Date'] = pd.to_datetime(combined_ndvi_df['Date'])
latest_ndvi_df = combined_ndvi_df.sort_values('Date').groupby(['x', 'y']).tail(1)
# If no acquisition date available, assume each (x, y) is unique (you can modify later)

 

ndvi_coords = latest_ndvi_df[['x', 'y']].values
ndvi_tree = cKDTree(ndvi_coords)

 

# Step 4: Assign PM2.5 and NDVI values to each edge and compute cost
alpha = 0.1  # Air quality weight
beta = 0.05  # NDVI reward weight

 

for u, v, data in G.edges(data=True):
    midpoint = ((u[0] + v[0]) / 2, (u[1] + v[1]) / 2)
    # Air quality
    dist, idx = aq_tree.query([midpoint[1], midpoint[0]])
    pm25 = latest_aq_df.iloc[idx]['PM25']
    
    # NDVI
    ndvi_dist, ndvi_idx = ndvi_tree.query([midpoint[0], midpoint[1]])  # x=longitude, y=latitude
    ndvi_value = latest_ndvi_df.iloc[ndvi_idx]['grid_code']  # Assuming 'grid_code' stores NDVI values
    
    # Assign values
    data['pm25'] = pm25
    data['ndvi'] = ndvi_value
    data['base_cost'] = data['weight'] * (1 + alpha * pm25 - beta * ndvi_value)
    data['cost'] = data['base_cost']

 

# Step 5: Get start and end locations
start_name = input("Enter the start location (e.g., Melrose Park): ")
end_name = input("Enter the end location (e.g., Hyde Park): ")

 

# Convert to coordinates using geopy
try:
    start_point = geocode_place(start_name)
    end_point = geocode_place(end_name)
    print(f"Start coordinates: {start_point}")
    print(f"End coordinates: {end_point}")
except ValueError as e:
    print(e)
    exit()  # Exit the program if locations can't be found



# Step 6: Find nearest nodes
nodes_list = list(G.nodes())
bike_tree = cKDTree([(lat, lon) for lon, lat in nodes_list])

 

_, start_idx = bike_tree.query((start_point[1], start_point[0]))
_, end_idx = bike_tree.query((end_point[1], end_point[0]))
start_node = nodes_list[start_idx]
end_node = nodes_list[end_idx]

 

# Step 7: Add virtual edges
def euclidean_distance(coord1, coord2):
    return sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

 

G.add_edge(start_point, start_node, weight=euclidean_distance(start_point, start_node), pm25=0, ndvi=0, base_cost=0, cost=0)
G.add_edge(end_point, end_node, weight=euclidean_distance(end_point, end_node), pm25=0, ndvi=0, base_cost=0, cost=0)

 

# Step 8: Yen's K-shortest paths with penalties
def yen_k_shortest_paths(graph, source, target, k, penalty_factor=2.0):
    paths = []
    for _ in range(k):
        path = nx.shortest_path(graph, source, target, weight='cost')
        paths.append(path)
        # Penalize reused edges
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            graph[u][v]['cost'] *= penalty_factor
    return paths

 

top_k_paths = yen_k_shortest_paths(G, start_point, end_point, 5)

 

# Step 9: Compute distances, PM2.5, and NDVI for each path
path_info = []
for path in top_k_paths:
    total_length = 0
    total_pm25 = 0
    total_ndvi = 0
    for i in range(len(path) - 1):
        edge_data = G.get_edge_data(path[i], path[i + 1])
        total_length += edge_data['weight']
        total_pm25 += edge_data['pm25']
        total_ndvi += edge_data['ndvi']
    avg_pm25 = total_pm25 / (len(path) - 1)
    avg_ndvi = total_ndvi / (len(path) - 1)
    path_info.append({'path': path, 'distance': total_length / 1000, 'pm25': avg_pm25, 'ndvi': avg_ndvi})

# Step 10: Ask user for ranking criteria
print("\nChoose the ranking criteria:")
print("1. Shortest path")
print("2. Least polluted path (lowest PM2.5)")
print("3. Greenest path (highest NDVI)")

 

criteria_choice = input("\nEnter the number corresponding to your choice (1, 2, or 3): ")

 

# Step 11: Sort paths based on the chosen criterion
if criteria_choice == "1":
    # Sort by shortest path
    path_info.sort(key=lambda x: x['distance'])
    criteria = "Shortest Path"
elif criteria_choice == "2":
    # Sort by least polluted (lowest PM2.5)
    path_info.sort(key=lambda x: x['pm25'])
    criteria = "Least Polluted Path"
elif criteria_choice == "3":
    # Sort by greenest (highest NDVI)
    path_info.sort(key=lambda x: -x['ndvi'])
    criteria = "Greenest Path"
else:
    print("Invalid choice! Exiting.")
    exit()

 

# Step 12: Print path details
print(f"\nRanking based on: {criteria}")
for idx, info in enumerate([path_info[0]] + path_info[1:]):
    label = "Best" if idx == 0 else f"Path {idx+1}"
    print(f"{label}: {info['distance']:.2f} km, Avg PM2.5: {info['pm25']:.2f}, Avg NDVI: {info['ndvi']:.2f}")

 

# Step 13: Visualization
m = folium.Map(location=[(start_point[1] + end_point[1]) / 2, (start_point[0] + end_point[0]) / 2], zoom_start=12)

 

# Draw network edges
for u, v, data in G.edges(data=True):
    coords = [(lat, lon) for lon, lat in [u, v]]
    folium.PolyLine(coords, color='lightgray', weight=1, opacity=0.3).add_to(m)

 

# Draw paths
colors = ['lightgreen', 'blue', 'orange', 'purple', 'brown']
for idx, info in enumerate(path_info):
    coords = [(lat, lon) for lon, lat in info['path']]
    folium.PolyLine(coords, color=colors[idx], weight=5, opacity=0.8, popup=f"Path {idx+1}: {info['distance']:.2f} km, PM2.5: {info['pm25']:.2f}, NDVI: {info['ndvi']:.2f}").add_to(m)

 

# Add start/end markers
folium.Marker(location=(start_point[1], start_point[0]), popup="Start Location", icon=folium.Icon(color='green')).add_to(m)
folium.Marker(location=(end_point[1], end_point[0]), popup="End Location", icon=folium.Icon(color='red')).add_to(m)

 

# Display the mapp
m


